# Coding the DuckChain 🦆

In this notebook, we'll build a simple blockchain in **Python**, one step at a time.

We'll use code to explore:

1. **Cryptographic Hashing** — creating digital fingerprints with SHA-256
2. **Hash Timing** — seeing how fast hashes can be calculated
3. **Blocks** — storing data and linking blocks with hashes
4. **DuckChain** — combining blocks into a blockchain
5. **Integrity** — detecting when blockchain data has been changed
6. **Consensus** — simulating a network that agrees on new blocks

The examples build on each other, from a single hash to a small blockchain network.

> **Goal:** Understand how a blockchain works by building one.

## 1. Cryptographic Hashing

A **cryptographic hash** creates a digital fingerprint of data.

In this example we will:

- Hash the word `"hello"` using **SHA-256**
- Hash a slightly more complex Python list
- Change one character and observe the resulting hash
- Hash a larger mock "duck image"

A useful cryptographic hash has an important property:

> **A very small change in the input produces a very different hash.**

Run the example once using SHA-256. Then change `HASH_ALGORITHM` to `"sha512"` and run it again.

In [ ]:
# hash.py - demonstrate hashing with different SHA algorithms

import hashlib
import json

# Define the hash algorithm to use
HASH_ALGORITHM = "sha256"  # Change this to "sha512" for SHA-512

# Function to perform hash
def sha_hash(text):
    return hashlib.new(HASH_ALGORITHM, text.encode("utf-8")).hexdigest()

# Hash simple object
simple_object = "hello"
simple_hash = sha_hash(simple_object)
print(f'{HASH_ALGORITHM} of "hello": {simple_hash}')

# Hash slightly more complex array object
array_object = ["a", "b", "c"]
array_hash = sha_hash(json.dumps(array_object))
print(f"{HASH_ALGORITHM} of array: {array_hash}")

# Show small variations cause big changes in resulting hash
variant_object = "hellO"
variant_hash = sha_hash(variant_object)
print(f'{HASH_ALGORITHM} of "hellO": {variant_hash}')

# Create a 64x64 "duck image" as a mock data structure and hash it
duck_image = [["yellow"] * 64 for _ in range(64)]
duck_image_data = {
    "name": "ducky",
    "color": "yellow",
    "weight": "1kg",
    "imageData": duck_image
}
duck_hash = sha_hash(json.dumps(duck_image_data))
print(f"{HASH_ALGORITHM} of duck object: {duck_hash}")

## 2. Timing Cryptographic Hashes

Hash functions are used constantly in blockchain systems.

This example calculates the same type of hash many times and measures:

- How long the experiment takes
- Approximately how many hashes the computer calculates per second

We will test three inputs:

1. A simple string
2. A small array
3. Our larger mock duck image

Try changing:

- `HASH_ALGORITHM` from `"sha256"` to `"sha512"`
- `ITERATIONS` to a larger or smaller value

**Question:** Does a larger input take noticeably longer to hash?

In [ ]:
# hashTiming.py - demonstrate and time SHA hashing

import hashlib
import json
import time

# Define the hash algorithm and number of iterations
HASH_ALGORITHM = "sha256"
ITERATIONS = 100_000

# Function to perform hash
def sha_hash(text):
    return hashlib.new(HASH_ALGORITHM, text.encode("utf-8")).hexdigest()

# Timing function for hashing
def time_hashing(obj, experiment_name):
    print(f"Running experiment: {experiment_name}")
    start_time = time.time()

    for i in range(ITERATIONS):
        sha_hash(obj + str(i))

    elapsed_time = time.time() - start_time
    hashes_per_second = ITERATIONS / elapsed_time
    print(f"Time for {ITERATIONS:,} hashes: {elapsed_time:.3f} seconds")
    print(f"Estimated hashes/sec: {hashes_per_second:,.0f}\n")

# Hash and time simple object
simple_object = "hello"
time_hashing(simple_object, "Simple Object")

# Hash and time slightly more complex array object
array_object = json.dumps(["a", "b", "c"])
time_hashing(array_object, "Array Object")

# Create and time a 64x64 mock "duck image"
duck_image = [["yellow"] * 64 for _ in range(64)]
duck_image_data = json.dumps({
    "name": "ducky", "color": "yellow",
    "weight": "1kg", "imageData": duck_image
})
time_hashing(duck_image_data, "Duck Image Object")

## 3. Linking Two Blocks

Now we use SHA-256 to build a simple **Block**.

Each block contains:

- `index` — its position
- `data` — the information stored in it
- `previous_hash` — the hash of the previous block
- `hash` — its own digital fingerprint

We will create two blocks:

**Block 0** is the special **Genesis Block**.

**Block 1** stores the Genesis Block's hash in its `previous_hash`.

That simple reference is what begins turning individual blocks into a **blockchain**.

> Look at `second_block.previous_hash` and `genesis_block.hash`.  
> They should be identical.

In [ ]:
# blocksTwo.py - create two cryptographically linked blocks

import hashlib
import json

# Function to perform SHA-256 hash
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Block class defining properties and behaviors of a block
class Block:
    # Method to create a new Block
    def __init__(self, index, data, previous_hash):
        self.index = index                  # Position of block in the chain
        self.data = data                    # Data stored in the block
        self.previous_hash = previous_hash  # Hash of the previous block
        self.hash = self.calculate_hash()   # Hash for this block

    # Method to calculate the hash of the block
    def calculate_hash(self):
        block_text = (
            str(self.index) +
            json.dumps(self.data, sort_keys=True) +
            str(self.previous_hash)
        )
        return sha256(block_text)

# Create first block (genesis block)
genesis_block = Block(
    0, {"transactions": ["Genesis Block Quack Quack!"]}, "0"
)
print("Genesis Block:", vars(genesis_block))

# Create second block and link it to the genesis block
second_block = Block(
    1,
    {"transactions": [
        "Donald Duck bought bread.",
        "Daisy Duck sold flowers."
    ]},
    genesis_block.hash
)
print("\nSecond Block:", vars(second_block))

# Check the cryptographic link
print("\nBlocks linked:",
      second_block.previous_hash == genesis_block.hash)

## 4. Building DuckChain

Two blocks are useful, but we want a program that can keep adding blocks.

We will add a **Blockchain** class that manages a list called `chain`.

The Blockchain class will:

- Automatically create the Genesis Block
- Find the latest block
- Add a new block
- Set the new block's `previous_hash`
- Recalculate the new block's hash

Then we'll add three duck transactions.

### Watch what happens when a block is added

The new block receives the **hash of the latest block** as its `previous_hash`.

That is the chain.

In [ ]:
# duckchain1.py - build a simple duck-themed blockchain

import hashlib
import json

# Function to perform SHA-256 hash
def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# This class defines a Block object destined for a Blockchain
class Block:
    def __init__(self, index, data):
        self.index = index                  # Block index
        self.data = data                    # Data stored in the block
        self.previous_hash = None           # Hash of the previous block
        self.hash = self.calculate_hash()   # Calculate hash for this block

    # Method to calculate the hash of a block
    def calculate_hash(self):
        block_text = (
            str(self.index) +
            json.dumps(self.data, sort_keys=True) +
            str(self.previous_hash)
        )
        return sha256(block_text)

# This class defines a blockchain object
class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    # Method to create the genesis block
    def create_genesis_block(self):
        return Block(0, "Genesis Block Quack Quack!")

    # Fetch the last block added to the chain
    def get_latest_block(self):
        return self.chain[-1]

    # Add a new block to the chain
    def add_block(self, new_block):
        new_block.previous_hash = self.get_latest_block().hash
        new_block.hash = new_block.calculate_hash()
        self.chain.append(new_block)

# Create and test the DuckChain
duckchain = Blockchain()
duckchain.add_block(Block(
    1, {"transaction": "Donald Duck sent 5 quackers to Daffy Duck"}
))
duckchain.add_block(Block(
    2, {"transaction": "Daffy Duck sent 3 quackers to Daisy Duck"}
))
duckchain.add_block(Block(
    3, {"transaction": "Daisy Duck sent 7 quackers to Scrooge McDuck"}
))

# Print the entire duck-themed blockchain
for block in duckchain.chain:
    print(vars(block))

## 5. Checking the Integrity of DuckChain

Now we give DuckChain a way to determine whether its history has been changed.

For each block, we perform **two tests**.

### Test 1 — Is the block itself intact?

Recalculate its hash.

If the new hash is different from the stored hash, someone changed the block.

### Test 2 — Is the chain still connected?

Compare the block's `previous_hash` with the actual hash of the block before it.

If they differ, the chain has been broken.

Then we'll try to cheat:

1. Change Donald's transaction from `5` quackers to `1000`
2. Recalculate that block's hash to cover our tracks
3. Ask DuckChain if everything is valid

**Prediction:** Did recalculating the hacked block's hash fix the chain?

In [ ]:
# duckchain2.py - add an integrity check to DuckChain

import hashlib
import json

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

class Block:
    def __init__(self, index, data):
        self.index = index
        self.data = data
        self.previous_hash = None
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_text = (
            str(self.index) +
            json.dumps(self.data, sort_keys=True) +
            str(self.previous_hash)
        )
        return sha256(block_text)

class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(0, "Genesis Block Quack Quack!")

    def get_latest_block(self):
        return self.chain[-1]

    def add_block(self, new_block):
        new_block.previous_hash = self.get_latest_block().hash
        new_block.hash = new_block.calculate_hash()
        self.chain.append(new_block)

    # Method to check if the chain is valid
    def is_chain_valid(self):
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i - 1]

            # Test 1: Has the current block been changed?
            if current_block.hash != current_block.calculate_hash():
                return False

            # Test 2: Is it still linked to the previous block?
            if current_block.previous_hash != previous_block.hash:
                return False

        return True

# Create and test the DuckChain
duckchain = Blockchain()
duckchain.add_block(Block(
    1, {"transaction": "Donald Duck sent 5 quackers to Daffy Duck"}
))
duckchain.add_block(Block(
    2, {"transaction": "Daffy Duck sent 3 quackers to Daisy Duck"}
))
duckchain.add_block(Block(
    3, {"transaction": "Daisy Duck sent 7 quackers to Scrooge McDuck"}
))

print("Initial DuckChain valid:", duckchain.is_chain_valid())

# Tamper with Block 1 and recalculate its hash to cover our tracks
duckchain.chain[1].data = {
    "transaction": "Donald Duck sent 1000 quackers to Daffy Duck"
}
duckchain.chain[1].hash = duckchain.chain[1].calculate_hash()

# Check blockchain validity
print("After tampering:", duckchain.is_chain_valid())

## 6. Network Consensus

A blockchain network has multiple participants that must agree before a new block is accepted.

In our simplified DuckChain network:

- Each `Admin` represents a network participant
- A new block is proposed
- Admins approve the block by signing its hash
- The network counts the valid approvals
- The block is added only when enough Admins agree

For DuckChain, we'll use a simple consensus rule:

> **At least 2 of 3 Admins must approve the proposed block.**

Our **teaching signature** connects an Admin's approval to the exact hash of the proposed block. Real blockchain systems use public/private key digital signatures.

In [ ]:
# duckchain3.py - simulate a blockchain network and consensus

import hashlib
import json

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

class Block:
    def __init__(self, index, data, previous_hash):
        self.index = index
        self.data = data
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()
        self.signatures = {}

    def calculate_hash(self):
        block_text = (
            str(self.index) +
            json.dumps(self.data, sort_keys=True) +
            str(self.previous_hash)
        )
        return sha256(block_text)

class Blockchain:
    def __init__(self):
        self.chain = [Block(0, "Genesis Block Quack Quack!", "0")]

    def get_latest_block(self):
        return self.chain[-1]

    def add_block(self, block):
        self.chain.append(block)

# Create a teaching signature for an Admin's approval
def sign(admin, block_hash):
    return sha256(admin + block_hash)

# Record an Admin's approval of this exact block
def vote(admin, block):
    block.signatures[admin] = sign(admin, block.hash)

# Check that the block is intact and has enough valid approvals
def has_consensus(block, admins, required_signatures):
    if block.hash != block.calculate_hash():
        return False

    approvals = 0
    for admin in admins:
        expected = sign(admin, block.hash)
        if block.signatures.get(admin) == expected:
            approvals += 1

    return approvals >= required_signatures

# Create DuckChain and our network Admins
duckchain = Blockchain()
admins = ["Huey", "Dewey", "Louie"]
required_signatures = 2

# Propose a new block linked to the latest block
new_block = Block(
    1,
    {"transaction": "Donald Duck sent 5 quackers to Daisy Duck"},
    duckchain.get_latest_block().hash
)

# Admins review and approve the proposed block
vote("Huey", new_block)
vote("Dewey", new_block)

# Add the block only if the network reaches consensus
if has_consensus(new_block, admins, required_signatures):
    duckchain.add_block(new_block)
    print("Consensus reached. Block added to DuckChain.")
else:
    print("Consensus not reached. Block rejected.")

print("Approvals:", list(new_block.signatures.keys()))
print("Blocks in DuckChain:", len(duckchain.chain))